### **Notebook #2**

### Libraries

In [11]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import statsmodels.api as sm
import math
import requests
import json
import time
from datetime import datetime, timedelta, timezone
import bs4 as bs
import yfinance as yf
import datetime
from scipy.stats import norm
from collections import Counter


from ipywidgets import interact, IntSlider, Checkbox
from functools import lru_cache
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
pd.set_option('display.max_colwidth', None)

## Task #2 : Social media coverage of press campaigns
- Use names of targeted firms and NGO + date of the campaign to see if social
 media talked about the story (Twitter, Reddit, Facebook, etc.)
- Requires translation + sentiment analysis

In [10]:
df = pd.read_csv('events_date_ngos_company_wide.csv')
df.head()

,date,isin_corporate_name_cleaned,company_parent_country,target_country1,target_country2,target_country3,target_country4,target_country5,target_country6,ngo_name1,ngo_name2,ngo_name3,ngo_name4,ngo_name5
0,2010-12-23,Deutsche Bank,Germany,Germany,NaN,NaN,NaN,NaN,NaN,Urgewald,NaN,NaN,NaN,NaN
1,2010-12-23,Commerzbank,Germany,Germany,NaN,NaN,NaN,NaN,NaN,Urgewald,NaN,NaN,NaN,NaN
2,2010-12-23,UniCredit,Italy,Germany,NaN,NaN,NaN,NaN,NaN,Urgewald,NaN,NaN,NaN,NaN
3,2010-12-23,UniCredit,Italy,Germany,NaN,NaN,NaN,NaN,NaN,Urgewald,NaN,NaN,NaN,NaN
4,2010-12-22,NaN,Sweden,Canada,NaN,NaN,NaN,NaN,NaN,Greenpeace Norway,NaN,NaN,NaN,NaN


We decided do scrape **Reddit** to identify if social media talked about the events connected to NGO campaigns.

For this we'll be looking for any posts published in the **10-days period after the campaign occured**, including the name of the **targeted company** and, optionally, the **name of the NGO**.

In [ ]:
# -----------------------------
# Config
# -----------------------------
SUBREDDITS = ["all"]  # or e.g., ["finance", "economics", "europe", "ukpolitics"]
MAX_PAGES_PER_QUERY = 10        # safety to avoid infinite loops
REQUESTS_PER_SECOND = 0.8       # be nice to Reddit; adjust if you get rate limited
USER_AGENT = "CampaignRedditScraper/1.0 (by u/YourUsername)"

INPUT_CSV  = "events_date_ngos_company_wide.csv"
OUTPUT_CSV = "reddit_matches_10day_window.csv"

In [ ]:
# -----------------------------
# Helpers
# -----------------------------
def clean_name(x):
    """Return a stripped string or None."""
    if pd.isna(x):
        return None
    s = str(x).strip()
    return s if s else None

def unique_names_from_row(row):
    """Collect present names from the row (company + NGO names)."""
    fields = ["isin_corporate_name_cleaned", "ngo_name1", "ngo_name2", "ngo_name3", "ngo_name4", "ngo_name5", "ngo_name6"]
    vals = [clean_name(row.get(f)) for f in fields if f in row]
    return sorted(set(v for v in vals if v))

def parse_date_ymd(s):
    """CSV shows 'YYYY-MM-DD'—parse to UTC-aware dt."""
    dt = datetime.strptime(s, "%Y-%m-%d")
    return dt.replace(tzinfo=timezone.utc)

def to_utc_datetime(created_utc):
    """Convert epoch seconds to UTC datetime."""
    return datetime.fromtimestamp(float(created_utc), tz=timezone.utc)

def search_reddit_for_name(name, start_dt, end_dt, subreddit="all", max_pages=10):
    """
    Use Reddit's search.json. We quote the name to improve precision.
    We paginate via 'after'. We still filter strictly by created_utc in [start_dt, end_dt].
    """
    url = "https://www.reddit.com/search.json"
    headers = {"User-Agent": USER_AGENT}

    # Search query: quoted phrase. If you want broader matches, drop quotes.
    q = f"\"{name}\""
    params = {
        "q": q,
        "sort": "new",
        "limit": 100,
        "t": "all",
    }
    # Restrict to a subreddit if not "all"
    if subreddit.lower() != "all":
        params["restrict_sr"] = 1
        # prepend subreddit: qualifier to improve result quality
        params["q"] = f"subreddit:{subreddit} {q}"

    after = None
    out = []
    pages = 0

    while pages < max_pages:
        if after:
            params["after"] = after

        resp = requests.get(url, headers=headers, params=params, timeout=30)
        resp.raise_for_status()
        data = resp.json().get("data", {})
        children = data.get("children", [])
        after = data.get("after")
        pages += 1

        for ch in children:
            d = ch.get("data", {})
            created = to_utc_datetime(d.get("created_utc"))
            if start_dt <= created <= end_dt:
                out.append({
                    "id": d.get("id"),
                    "subreddit": d.get("subreddit"),
                    "title": d.get("title"),
                    "author": d.get("author"),
                    "created_utc": created.isoformat(),
                    "score": d.get("score"),
                    "num_comments": d.get("num_comments"),
                    "url": d.get("url"),
                    "selftext": d.get("selftext", ""),
                    "permalink": f"https://reddit.com{d.get('permalink')}",
                })

        # Heuristic exit: if the newest post on this page is already older than the window start,
        # further pages will likely be even older; break early.
        if children:
            newest_created = to_utc_datetime(children[0]["data"]["created_utc"])
            if newest_created < start_dt:
                break

        if not after:
            break

        time.sleep(1.0 / REQUESTS_PER_SECOND)

    return out

In [ ]:
# -----------------------------
# Main
# -----------------------------
df = pd.read_csv(INPUT_CSV)

# Ensure required columns exist
needed = {"date", "isin_corporate_name_cleaned", "ngo_name1", "ngo_name2", "ngo_name3", "ngo_name4", "ngo_name5", "ngo_name6"}
missing = [c for c in needed if c not in df.columns]
if missing:
    raise ValueError(f"Missing expected columns in CSV: {missing}")

results = []
seen_ids = set()  # global de-dup by Reddit post id
cache = {}        # cache[(name, start_iso, end_iso, subreddit)] -> list of posts

for idx, row in df.iterrows():
    # Build 10-day window centered on 'date' (±5 days)
    center = parse_date_ymd(str(row["date"]))
    start_dt = center
    end_dt   = center + timedelta(days=10)

    names = unique_names_from_row(row)
    if not names:
        continue

    for name in names:
        for sr in SUBREDDITS:
            key = (name, start_dt.date().isoformat(), end_dt.date().isoformat(), sr)
            if key not in cache:
                try:
                    cache[key] = search_reddit_for_name(name, start_dt, end_dt, subreddit=sr, max_pages=MAX_PAGES_PER_QUERY)
                except requests.RequestException as e:
                    print(f"[warn] request failed for {key}: {e}")
                    cache[key] = []

            for post in cache[key]:
                if post["id"] in seen_ids:
                    continue
                seen_ids.add(post["id"])

                results.append({
                    # campaign metadata
                    "campaign_row": idx,
                    "campaign_date": center.date().isoformat(),
                    "company": clean_name(row["isin_corporate_name_cleaned"]),
                    "matched_name": name,
                    # reddit fields
                    **post
                })

In [ ]:
# Save
out_df = pd.DataFrame(results)
out_df.to_csv(OUTPUT_CSV, index=False)

print(f"Saved {len(out_df)} posts to {OUTPUT_CSV}")